In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
## Parte 2. Revision Del Dataset

# Carga el dataset con pandas.
ventas = pd.read_csv("ventas_ecommerce_limpio.csv")

#Realiza lo siguiente:

In [3]:
# 1. Muestra las primeras filas.
ventas.head(5)

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [4]:
# 2. Revisa las columnas.
ventas.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [5]:
# 3. Muestra cuantas filas y columnas tiene.
ventas.shape

(60, 10)

In [6]:
# 4. Revisa si hay valores nulos.
ventas.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [8]:
# 5. Verifica que exista la columna `total_venta`.
ventas["total_venta"].head(5)

0    250.0
1    650.0
2    900.0
3    800.0
4    360.0
Name: total_venta, dtype: float64

In [10]:
# 6. Verifica que `total_venta` coincida con `cantidad * precio_unitario`.
ventas["total_calculado"] = ventas["cantidad"] * ventas["precio_unitario"]
ventas[ventas["total_venta"] != ventas["total_calculado"]]

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado


In [11]:
# 7. Escribe una observacion breve sobre el estado del dataset.
# El data set muestra que la cantidad por el precio unitario coinciden con total_venta

In [15]:
## Parte 3. Variable Objetivo
# 1. Puedes usar una funcion normal o una lambda.
ventas["venta_alta"] = ventas["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)

In [17]:
# 2. Cuenta cuantas ventas quedaron como 1 y cuantas como 0.
ventas["venta_alta"].value_counts()
# Fueron 40 ventas altas y 20 ventas no altas

venta_alta
1    40
0    20
Name: count, dtype: int64

In [14]:
# Por que venta_alta es la variable objetivo? R= Porque es el resultado que se espera predecir para nuevos datos que cumplan esa condicion

In [21]:
## Parte 4. Variables De Entrada
# 1. Crea `X`.
X = ventas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [22]:
# 2. Crea `y`.
y = ventas["venta_alta"]

In [23]:
# 3. Convierte variables categoricas con `pd.get_dummies()`.
X = pd.get_dummies(X)

In [24]:
# 4. Guarda la lista de columnas generadas.
columnas_modelo = X.columns.tolist()

In [25]:
# 5. Muestra las primeras filas de `X` despues de `get_dummies()`.
X.head(5)

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


In [28]:
# Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta? R= 
# R= Porque los datos en general vienen de ahi de total venta, si usaramos total venta practicamente le estariamos dando las respuestas al modelo y este
# no aprenderia nada, practicamente seria una fuga de datos y no estaria aprendiendo.

In [27]:
## Parte 5. Entrenamiento Y Evaluacion
# Entrena un modelo con `DecisionTreeClassifier`.

In [26]:
# 1. Divide los datos en entrenamiento y prueba.
# 2. Usa 80% entrenamiento y 20% prueba.
# 3. Usa `random_state=42`.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
# 4. Entrena el modelo.
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [30]:
# 5. Genera predicciones con los datos de prueba.
predicciones = modelo.predict(X_test)

In [31]:
# 6. Calcula exactitud.
exactitud = accuracy_score(y_test, predicciones)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [32]:
# 7. Muestra matriz de confusion.
matriz = confusion_matrix(y_test, predicciones)
print(matriz)

[[4 0]
 [0 8]]


In [33]:
# 8. Crea una tabla llamada `resultados_prueba` con: valor_real, prediccion, coincide
resultados_prueba = pd.DataFrame({
    "valor_real": y_test,
    "prediccion": predicciones
})

In [35]:
# 9. Cuenta cuantos aciertos y cuantos errores hubo.
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]
resultados_prueba["coincide"].value_counts()

aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

print("Aciertos", len(aciertos))
print("Errores: ", len(errores))

Aciertos 12
Errores:  0


In [36]:
# Preguntas obligatorias:
# 1. Cual fue la exactitud? - Fue de 1.0
# 2. Cuantos aciertos tuvo el modelo? - 12 aciertos
# 3. Cuantos errores tuvo el modelo? - 0 aciertos
# 4. Que indica la matriz de confusion? - indica que hubo 4 resultados correctos y 0 incorrectos al igual que otros 0 incorrectos y 8 correctos
# 5. Una buena exactitud significa que el modelo ya es perfecto? Explica. 
# Si pero siempre y cuando sean mas datos, como en este caso trabajamos con pocos datos, la exactictud no podria ser tan confiable que digamos.

In [37]:
## Parte 6. Guardar Modelo Y Columnas
# 1. Usa `joblib`.
# 2. Guarda el modelo entrenado.
# 3. Guarda la lista de columnas usadas durante el entrenamiento.
# 4. Verifica que los archivos aparezcan en tu carpeta.

joblib.dump(modelo, "modelo_examen_venta_alta.pkl")
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")

['columnas_examen_modelo.pkl']

In [ ]:
# Preguntas obligatorias:
# 1. Para que sirve guardar el modelo? - Para que en un futuro cuando queramos hacer predicciones usemos este mismo modelo que aprendio a clasificar ventas
# si es que agregaramos nuevos datos, el modelo podria aprender incluso con estos nuevos datos sin tener que darle todo el contexto anteriormente porque
# el aprendizaje se pierde al cerrar la sesion.

# 2. Para que sirve guardar las columnas del entrenamiento?
# lo mismo para guardar el modelo, guardamos las columnas de los datos ya predecidos al momento de predecir nuevos datos, este pueda hacerlo en base a los
# nuevos datos sin necesidad que hacer todo lo anterior.

# 3. Que problema puede aparecer si no guardas las columnas?
# Al no tener las mismas columnas alineadas por asi decirlo, el modelo no seria capaz de predecir con nuevos datos y generaria error al momento de compilar
# si se quisiera predecir nuevamente.

In [ ]:
## Parte 7. Ventas Nuevas

In [44]:
## Parte 8. Cargar Modelo Y Predecir
# Usa el modelo guardado para predecir tus ventas nuevas.
# Indicaciones:

In [42]:
# 1. Carga `modelo_examen_venta_alta.pkl`.
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")

In [43]:
# 2. Carga `columnas_examen_modelo.pkl`.
columnas_modelo = joblib.load("columnas_examen_modelo.pkl")

In [45]:
# 3. Carga `examen_ventas_nuevas.csv`.
ventas_nuevas = pd.read_csv("examen_ventas_nuevas.csv")
ventas_nuevas.head(10)

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad
0,1061,2026-07-01,Ania Garcia,Mouse Gamer,Computacion,1,1500.0,Efectivo,Cuernavaca
1,1062,2026-07-01,Andrew Torres,Teclado Gamer,Computacion,1,2000.0,Tarjeta,Ayala
2,1063,2026-07-02,Sandra Ramoz,Audifonos Gamer,Electronica,1,3200.0,Tarjeta,Temixco
3,1064,2026-07-02,Perla Olmeda,Webcam PRO MAX,Accesorios,1,1350.0,Efectivo,Cuernavaca
4,1065,2026-07-03,Laura Marquez,Cable HDMI NVIDIA,Accesorios,2,200.0,Efectivo,Jiutepec
5,1066,2026-07-03,Carlos Trejo,Memoria USB KINGSTON,Accesorios,1,500.0,Tarjeta,Xochitepec
6,1067,2026-07-04,Monica Solis,Mouse Pad MAC,Accesorios,1,800.0,Efectivo,Temixco
7,1068,2026-07-04,Jorge Linarez,Cargador 34 Volts,Electronica,3,250.0,Tarjeta,Cuernavaca
8,1069,2026-07-05,Valeria Castillo,Funda Tablet Micky mouse,Accesorios,1,1000.0,Efectivo,Jiutepec
9,1070,2026-07-05,Daniel Flores,Soporte Celular Xiaomi,Accesorios,1,1000.0,Efectivo,Temixco


In [47]:
# 4. Selecciona las mismas variables de entrada usadas en entrenamiento.
X_nuevas = ventas_nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [48]:
# 5. Aplica `pd.get_dummies()`.
X_nuevas = pd.get_dummies(X_nuevas)

In [49]:
# 6. Alinea columnas con `reindex()`.
X_nuevas = X_nuevas.reindex(columns=columnas_modelo, fill_value=0)

In [53]:
# 7. Genera predicciones.
# 8. Agrega la columna: prediccion_venta_alta
ventas_nuevas["prediccion_venta_alta"] = modelo_cargado.predict(X_nuevas)


In [52]:
# 9. Agrega la columna: interpretacion_prediccion
ventas_nuevas["interpretacion_prediccion"] = ventas_nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

In [55]:
ventas_nuevas.head(10)

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion
0,1061,2026-07-01,Ania Garcia,Mouse Gamer,Computacion,1,1500.0,Efectivo,Cuernavaca,1,Venta alta
1,1062,2026-07-01,Andrew Torres,Teclado Gamer,Computacion,1,2000.0,Tarjeta,Ayala,1,Venta alta
2,1063,2026-07-02,Sandra Ramoz,Audifonos Gamer,Electronica,1,3200.0,Tarjeta,Temixco,1,Venta alta
3,1064,2026-07-02,Perla Olmeda,Webcam PRO MAX,Accesorios,1,1350.0,Efectivo,Cuernavaca,0,Venta no alta
4,1065,2026-07-03,Laura Marquez,Cable HDMI NVIDIA,Accesorios,2,200.0,Efectivo,Jiutepec,0,Venta no alta
5,1066,2026-07-03,Carlos Trejo,Memoria USB KINGSTON,Accesorios,1,500.0,Tarjeta,Xochitepec,0,Venta no alta
6,1067,2026-07-04,Monica Solis,Mouse Pad MAC,Accesorios,1,800.0,Efectivo,Temixco,0,Venta no alta
7,1068,2026-07-04,Jorge Linarez,Cargador 34 Volts,Electronica,3,250.0,Tarjeta,Cuernavaca,1,Venta alta
8,1069,2026-07-05,Valeria Castillo,Funda Tablet Micky mouse,Accesorios,1,1000.0,Efectivo,Jiutepec,0,Venta no alta
9,1070,2026-07-05,Daniel Flores,Soporte Celular Xiaomi,Accesorios,1,1000.0,Efectivo,Temixco,0,Venta no alta


In [56]:
#10. Guarda el resultado como: examen_predicciones.csv
ventas_nuevas.to_csv("examen_predicciones.csv", index=False)

In [57]:
# Que podria pasar si no usas reindex antes de predecir? - No se podrian realizar las predicciones debido a que las columnas no se hubieran alineado
# con los nuevos datos que se predijeron despues

In [58]:
## Parte 9. Auditoria Del Modelo
ventas_nuevas["total_estimado"] = ventas_nuevas["cantidad"] * ventas_nuevas["precio_unitario"]

In [74]:
ventas_nuevas["venta_alta_real_estimada"] = ventas_nuevas["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)
ventas_nuevas["venta_alta_real_estimada"].value_counts()

venta_alta_real_estimada
1    6
0    4
Name: count, dtype: int64

In [60]:
ventas_nuevas["coincide"] = ventas_nuevas["prediccion_venta_alta"] == ventas_nuevas["venta_alta_real_estimada"]

In [61]:
ventas_nuevas["coincide"].value_counts()

coincide
True     6
False    4
Name: count, dtype: int64

In [65]:
# 5. Cuenta cuantas predicciones coincidieron y cuantas no. - Coincidieron 6 y 4 no.
# 6. Filtra las ventas que no coincidieron.
errores = ventas_nuevas[ventas_nuevas["coincide"] == False]
errores

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide
3,1064,2026-07-02,Perla Olmeda,Webcam PRO MAX,Accesorios,1,1350.0,Efectivo,Cuernavaca,0,Venta no alta,1350.0,1,False
7,1068,2026-07-04,Jorge Linarez,Cargador 34 Volts,Electronica,3,250.0,Tarjeta,Cuernavaca,1,Venta alta,750.0,0,False
8,1069,2026-07-05,Valeria Castillo,Funda Tablet Micky mouse,Accesorios,1,1000.0,Efectivo,Jiutepec,0,Venta no alta,1000.0,1,False
9,1070,2026-07-05,Daniel Flores,Soporte Celular Xiaomi,Accesorios,1,1000.0,Efectivo,Temixco,0,Venta no alta,1000.0,1,False


In [72]:
# 7. Revisa si los errores estan cerca del limite de 1000.
ventas_nuevas["distancia_a_1000"] = (ventas_nuevas["total_estimado"] - 1000).abs()
cerca_limite = ventas_nuevas[ventas_nuevas["distancia_a_1000"] <= 200]
cerca_limite

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
6,1067,2026-07-04,Monica Solis,Mouse Pad MAC,Accesorios,1,800.0,Efectivo,Temixco,0,Venta no alta,800.0,0,True,200.0
8,1069,2026-07-05,Valeria Castillo,Funda Tablet Micky mouse,Accesorios,1,1000.0,Efectivo,Jiutepec,0,Venta no alta,1000.0,1,False,0.0
9,1070,2026-07-05,Daniel Flores,Soporte Celular Xiaomi,Accesorios,1,1000.0,Efectivo,Temixco,0,Venta no alta,1000.0,1,False,0.0


In [67]:
# 8. Guarda de nuevo `examen_predicciones.csv` con las columnas de auditoria.
ventas_nuevas.to_csv("examen_predicciones.csv", index=False)

In [ ]:
# Preguntas obligatorias:

# 1. Cuantas ventas nuevas evaluaste? - 10
# 2. Cuantas fueron predichas como venta alta? - 6
# 3. Cuantas fueron predichas como venta no alta? - 4
# 4. Cuantas coincidieron con la regla manual? - lo mimsmo 6 si coinciden y 4 no.
# 5. Cuantas no coincidieron? - 4
# 6. Que ventas no coincidieron? - la 3, 7 , 8 y 9
# 7. Los errores estuvieron cerca del limite de 1000? - Si algunos
# 8. Que paso con la categoria nueva? - Si se logro crear 
# 9. Que paso con la ciudad nueva? - tambien se logro crear